# Évaluation CL — HDC (D=1024) — Dataset 3 PRONOSTIA — by_condition

| Champ | Valeur |
|-------|--------|
| **Modèle** | HDC Hyperdimensional Computing (D=1024, 2 048 prototypes) |
| **Dataset** | FEMTO PRONOSTIA — roulements à billes industriels |
| **Scénario** | by_condition : Condition 1 → Condition 2 → Condition 3 (3 tâches) |
| **Expérience** | exp_051 — voir experiments/exp_051_hdc_pronostia_by_condition/config_snapshot.yaml |
| **Sprint** | 10 — S10-06 |

> **Modèle supervisé non-neuronal** : accumulation additive d'hypervecteurs binarisés (INT8).  
> Aucun gradient, pas de momentum — conforme MCU.  
> Inférence : argmax de la similarité cosinus (via POPCOUNT/XOR sur MCU).  
> RAM = 14.2 Ko @ FP32 — dans le budget 64 Ko STM32N6.  
> AF = 0.045 — légère dégradation sur PRONOSTIA (vs. 0.0 sur monitoring).  
> **Gap 1** : résultat CL HDC sur données industrielles réelles de roulements (PRONOSTIA).

```bash
jupyter nbconvert --to notebook --execute \
    notebooks/cl_eval/pronostia_by_condition/hdc.ipynb \
    --output /tmp/hdc_pronostia_executed.ipynb --ExecutePreprocessor.timeout=600
```

In [ ]:
# Section 1 — Setup & imports
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Image, Markdown, display

# --- CWD navigation : notebook 3 niveaux de profondeur ---
_cwd = Path(".").resolve()
if _cwd.name == "pronostia_by_condition":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import (
    plot_accuracy_matrix,
    plot_confusion_matrix_grid,
    plot_forgetting_curve,
    plot_roc_curves_per_task,
    save_figure,
)
from src.evaluation.feature_space_plots import fit_pca2d, plot_feature_space_2d

# --- Chemins ---
EXP_DIR     = REPO_ROOT / "experiments/exp_051_hdc_pronostia_by_condition/results"
FIGURES_DIR = REPO_ROOT / "notebooks/figures/cl_evaluation/hdc/pronostia/by_condition"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH     = REPO_ROOT / "configs/hdc_pronostia_by_condition_config.yaml"
NPY_DIR         = REPO_ROOT / "data/raw/Pronostia dataset/binaries"
NORMALIZER_PATH = REPO_ROOT / "configs/pronostia_normalizer.yaml"

# --- Constantes ---
TASK_NAMES     = ["Condition 1 (1800rpm, 4000N)", "Condition 2 (1650rpm, 4200N)", "Condition 3 (1500rpm, 5000N)"]
MODEL_NAME     = "HDC"
DATA_AVAILABLE = NPY_DIR.exists() and CONFIG_PATH.exists()

print(f"REPO_ROOT         : {REPO_ROOT}")
print(f"EXP_DIR           : {EXP_DIR}")
print(f"FIGURES_DIR       : {FIGURES_DIR}")
print(f"NPY disponible    : {NPY_DIR.exists()}")
print(f"Config disponible : {CONFIG_PATH.exists()}")
print(f"Date exécution    : {datetime.now():%Y-%m-%d %H:%M}")

if not DATA_AVAILABLE:
    display(Markdown(
        "> ⚠️ **Ressources absentes** — Sections 5, 6, 7, 8 en mode dégradé (données synthétiques)."
    ))

In [ ]:
# Section 2 — Chargement des résultats exp_051
# HDC PRONOSTIA : metrics.json → cl_metrics est plat (pas de sous-clé "hdc")

metrics_path    = EXP_DIR / "metrics.json"
acc_matrix_path = EXP_DIR / "acc_matrix_hdc.npy"

metrics    = json.loads(metrics_path.read_text())
acc_matrix = np.load(acc_matrix_path, allow_pickle=True)

cl = metrics["cl_metrics"]

# Reconstruire la matrice acc numpy (remplacement null → NaN)
acc_matrix_json = np.array(
    [[v if v is not None else np.nan for v in row] for row in cl["acc_matrix"]],
    dtype=float,
)

aa       = cl["aa"]
af       = cl["af"]
bwt      = cl["bwt"]
ram_b    = cl["ram_peak_bytes"]
lat      = cl["inference_latency_ms"]
n_params = cl.get("n_params", 2048)
ram_int8 = cl.get("estimated_ram_int8_bytes", None)

print("=" * 55)
print(f"  Modèle         : HDC (D=1024, n_params={n_params})")
print(f"  AA             = {aa:.4f}")
print(f"  AF             = {af:.4f}")
print(f"  BWT            = {bwt:+.4f}")
print(f"  Forgetting/tâche: {[round(v, 4) for v in cl.get('forgetting_per_task', [])]}")
print(f"  RAM peak       = {ram_b} B ({ram_b/1024:.2f} Ko @ FP32)")
if ram_int8:
    print(f"  RAM INT8       = {ram_int8} B ({ram_int8/1024:.2f} Ko)")
print(f"  Latence        = {lat:.5f} ms")
print(f"  Budget 64 Ko   : {ram_b <= 65536}")
print("=" * 55)
print("\nMatrice acc (3×3) :")
print(acc_matrix_json)

In [ ]:
# Section 3 — Matrice d'accuracy (heatmap)
# HDC PRONOSTIA : AF=0.045 (contrairement au monitoring where AF=0.0)
# Les conditions opératoires génèrent un léger oubli malgré les prototypes additifs

fig = plot_accuracy_matrix(
    acc_matrix_json,
    task_names=TASK_NAMES,
    title=f"{MODEL_NAME} — pronostia/by_condition",
)
save_figure(fig, FIGURES_DIR / "acc_matrix.png")
display(Image(str(FIGURES_DIR / "acc_matrix.png")))

In [ ]:
# Section 4 — Courbe d'oubli par tâche

fig = plot_forgetting_curve(
    acc_matrix_json,
    task_names=TASK_NAMES,
    title=f"{MODEL_NAME} — Évolution accuracy par tâche (pronostia/by_condition)",
)
save_figure(fig, FIGURES_DIR / "forgetting_curve.png")
display(Image(str(FIGURES_DIR / "forgetting_curve.png")))

In [ ]:
# Section 5 — Rejeu du scénario CL (collecte preds_dict + proba_dict)
# HDC : accumulation des hypervecteurs → binarisation → inférence par similarité cosinus
# Score continu = (sim[1] - sim[0]) / (2*D) + 0.5  (pour courbes ROC)

preds_dict  = {}
proba_dict  = {}
X_tests_raw = []
y_tests_raw = []

def hdc_scores(model, X):
    from src.models.hdc.hdc_classifier import encode_observation
    scores = np.zeros(len(X), dtype=np.float32)
    for idx, sample in enumerate(X):
        H_obs = encode_observation(
            sample, model.H_level, model.H_pos, model.feature_bounds, model.n_levels, model.D,
        )
        sims = model.prototypes_bin.astype(np.float32) @ H_obs.astype(np.float32)
        scores[idx] = (sims[1] - sims[0]) / (2.0 * model.D) + 0.5
    return scores

if DATA_AVAILABLE:
    from src.data.pronostia_dataset import get_pronostia_dataloaders
    from src.models.hdc.hdc_classifier import HDCClassifier
    from src.utils.reproducibility import set_seed

    set_seed(42)

    # Charger la config HDC PRONOSTIA
    config = yaml.safe_load(CONFIG_PATH.read_text())

    tasks = get_pronostia_dataloaders(
        npy_dir=NPY_DIR,
        normalizer_path=NORMALIZER_PATH,
        batch_size=256,
        seed=42,
    )

    # Extraire les données de validation en numpy une seule fois
    for t in tasks:
        X_v = np.concatenate([b[0].numpy() for b in t["val_loader"]])
        y_v = np.concatenate([b[1].numpy().flatten() for b in t["val_loader"]])
        X_tests_raw.append(X_v)
        y_tests_raw.append(y_v)

    # Instancier HDCClassifier avec config PRONOSTIA
    model = HDCClassifier(config)

    for i, task in enumerate(tasks):
        domain = task.get("domain", f"condition_{task['task_id']}")
        print(f"\n--- Tâche {i + 1}/3 : {domain} ---")

        # Accumulation des hypervecteurs (pas de gradient, 1 passe)
        X_train = np.concatenate([b[0].numpy() for b in task["train_loader"]])
        y_train = np.concatenate([b[1].numpy().flatten() for b in task["train_loader"]]).astype(np.int64)
        err = model.update(X_train, y_train)
        model.on_task_end(task["task_id"], task["train_loader"])
        print(f"  Erreur train : {err:.4f}  (N={len(y_train)})")

        # Évaluation sur toutes les tâches vues jusqu'à i
        for j in range(i + 1):
            y_pred = model.predict(X_tests_raw[j]).astype(float)
            scores = hdc_scores(model, X_tests_raw[j])
            preds_dict[(i, j)] = (y_tests_raw[j], y_pred)
            proba_dict[(i, j)] = scores
            acc = (y_tests_raw[j] == y_pred).mean()
            print(f"  preds_dict[({i},{j})] → N={len(y_tests_raw[j])}, acc={acc:.4f}")

    print(f"\nScénario CL rejoué — {len(preds_dict)} évaluations collectées")

else:
    display(Markdown("> ⚠️ **Mode dégradé** — preds_dict synthétique depuis acc_matrix."))

    N_SYNTH = 500
    rng = np.random.default_rng(42)
    # PRONOSTIA : ~10% positifs
    y_synth = np.concatenate([np.zeros(int(N_SYNTH * 0.9)), np.ones(int(N_SYNTH * 0.1))])

    for i in range(3):
        for j in range(i + 1):
            noise = rng.normal(0, 0.15, len(y_synth))
            probas_synth = np.where(y_synth == 1, 0.60 + noise, 0.40 + noise).clip(0, 1)
            y_pred_synth = (probas_synth >= 0.5).astype(float)
            preds_dict[(i, j)] = (y_synth.copy(), y_pred_synth)
            proba_dict[(i, j)] = probas_synth.astype(np.float32)

    print("preds_dict synthétique créé (mode dégradé)")

In [ ]:
# Section 6 — Matrices de confusion par tâche (grille 3×3)

fig = plot_confusion_matrix_grid(
    preds_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
    threshold=0.5,
)
save_figure(fig, FIGURES_DIR / "confusion_matrix_grid.png")
display(Image(str(FIGURES_DIR / "confusion_matrix_grid.png")))

In [ ]:
# Section 7 — Courbes ROC par tâche
# HDC : scores reconstruits via (sim_class1 - sim_class0) normalisés
# PRONOSTIA : ~10% positifs → AUC-ROC particulièrement important

fig = plot_roc_curves_per_task(
    preds_dict,
    scores_dict=proba_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
)

fig.text(
    0.5, 0.01,
    f"AA exp_051 = {aa:.4f}  |  AF = {af:.4f}  |  BWT = {bwt:+.4f}",
    ha="center", fontsize=9, color="#444444",
)

save_figure(fig, FIGURES_DIR / "roc_curves.png")
display(Image(str(FIGURES_DIR / "roc_curves.png")))

In [ ]:
# Section 8 — Espace des features (PCA 2D)
# Coloré par condition (1/2/3) et par label (normal/faulty)

if DATA_AVAILABLE and len(X_tests_raw) == 3:
    X_all      = np.concatenate(X_tests_raw, axis=0)   # [N_total, 13]
    y_all      = np.concatenate(y_tests_raw, axis=0)
    domain_ids = np.concatenate([
        np.full(len(X_tests_raw[k]), k) for k in range(3)
    ])

    pca, X_proj = fit_pca2d(X_all)
    expl_var = pca.explained_variance_ratio_
    xlabel = f"PC1 ({expl_var[0]*100:.1f}%)"
    ylabel = f"PC2 ({expl_var[1]*100:.1f}%)"

    fig, ax = plt.subplots(figsize=(9, 7))

    plot_feature_space_2d(
        X_proj, y_all,
        title=f"{MODEL_NAME} — Espace features PCA 2D — pronostia/by_condition",
        ax=ax,
        domain_ids=domain_ids,
        alpha=0.25,
        s=6,
        xlabel=xlabel,
        ylabel=ylabel,
    )

    TASK_COLORS = ["#2196F3", "#FF9800", "#9C27B0"]
    for k, (name, color) in enumerate(zip(TASK_NAMES, TASK_COLORS)):
        mask = domain_ids == k
        cx, cy = X_proj[mask, 0].mean(), X_proj[mask, 1].mean()
        ax.annotate(
            f"C{k+1}",
            xy=(cx, cy),
            fontsize=11,
            fontweight="bold",
            color=color,
            ha="center",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
        )

    fig.tight_layout()
    save_figure(fig, FIGURES_DIR / "feature_space_pca.png")
    display(Image(str(FIGURES_DIR / "feature_space_pca.png")))

else:
    display(Markdown("> ⚠️ **Section 8 ignorée** — données non disponibles."))
    print("[SKIP] feature_space_pca.png")

## Discussion — Gap 1

Ces résultats sur FEMTO PRONOSTIA (données industrielles réelles de roulements à billes)
complètent les validations sur les datasets Kaggle (Equipment Monitoring, Pump Maintenance).

**Comparaison cross-dataset** :
- Dataset Monitoring (by_equipment, 3 tâches) : AA ≈ 0.8698, AF ≈ 0.0000
- **Dataset PRONOSTIA (by_condition, 3 tâches) : AA = 0.8051, AF = 0.0454** ← exp_051

**HDC sur PRONOSTIA vs. Monitoring** :
- Sur le monitoring, AF = 0.0 par design (prototypes additifs non-destructifs).
- Sur PRONOSTIA, AF = 0.045 — les conditions opératoires génèrent un drift de feature plus marqué
  que le changement de type d'équipement, ce qui perturbe la binarisation des prototypes.
- Cela suggère que l'encodage HDC (quantification + Hadamard) est sensible au drift de distribution
  dans l'espace original.

**Architecture-based (sans gradient)** :
- Aucun optimiseur, aucun état Adam — conforme MCU strict
- Inférence par POPCOUNT/XOR : portable en C sur Cortex-M55

`FIXME(gap1)` : ✅ Résultat CL HDC sur données industrielles réelles de roulements —
voir `docs/roadmap_phase1.md` section Sprint 10 pour la synthèse complète.

`TODO(arnaud)` : La section Discussion Gap 1 doit-elle inclure une référence bibliographique
au protocole IEEE PHM 2012 Challenge (`@PHM2012`) pour contextualiser dans la littérature ?

`TODO(dorra)` : La dimension D=1024 est-elle optimale pour PRONOSTIA (13 features) ?
Tester D=512 pour réduire la RAM à ~7 Ko.

In [ ]:
# Section 10 — Tableau récapitulatif + critères d'acceptation

ram_ko = ram_b / 1024

display(Markdown("### Résultats finaux — HDC — pronostia/by_condition (exp_051)"))

recap_table = f"""
| Modèle | AA ↑ | AF ↓ | BWT | RAM ↓ | Latence ↓ | n_params |
|--------|------|------|-----|-------|-----------|----------|
| {MODEL_NAME} | {aa:.4f} | {af:.4f} | {bwt:+.4f} | {ram_ko:.2f} Ko | {lat:.5f} ms | {n_params} |
"""
display(Markdown(recap_table))

print("=" * 60)
print("  NOTE SCIENTIFIQUE — Gap 2 (contrainte embarquée STM32N6)")
print("=" * 60)
print(f"  RAM = {ram_b} B = {ram_ko:.2f} Ko @ FP32")
if ram_int8:
    print(f"  RAM INT8 = {ram_int8} B = {ram_int8/1024:.2f} Ko @ INT8 binarisé")
print(f"  Budget STM32N6 : 65 536 B (64 Ko)")
print(f"  Marge disponible : {65536 - ram_b} B ({(65536 - ram_b)/1024:.1f} Ko)")
print()

print("=" * 60)
print("  NOTE SCIENTIFIQUE — Gap 3 (architecture-based, pas de replay)")
print("=" * 60)
print(f"  Prototypes additifs (INT32 accumulateur + INT8 binarisé)")
print(f"  Aucun gradient, aucun optimiseur — conforme MCU strict")
print(f"  AF={af:.4f} sur PRONOSTIA (vs. 0.0 sur monitoring — drift conditions plus marqué)")
print()

print("=" * 60)
print("  Critères d'acceptation (S10-06)")
print("=" * 60)
for fig_name in ["acc_matrix.png", "forgetting_curve.png", "confusion_matrix_grid.png",
                 "roc_curves.png", "feature_space_pca.png"]:
    status = "OK" if (FIGURES_DIR / fig_name).exists() else "MANQUANTE"
    print(f"  [{status}] {fig_name}")

print()
print(f"  [{'OK' if abs(aa - 0.8051) < 0.005 else 'WARN'}] AA     = {aa:.4f}  (attendu ≈ 0.8051)")
print(f"  [{'OK' if abs(af - 0.0454) < 0.005 else 'WARN'}] AF     = {af:.4f}  (attendu ≈ 0.0454)")
print(f"  [{'OK' if abs(bwt + 0.0454) < 0.005 else 'WARN'}] BWT    = {bwt:+.4f} (attendu ≈ -0.0454)")
print(f"  [{'OK' if ram_b <= 65536 else 'FAIL'}] RAM    = {ram_ko:.2f} Ko (contrainte ≤ 64 Ko)")
print(f"  [{'OK' if lat < 100.0 else 'WARN'}] Latence= {lat:.5f} ms (contrainte ≤ 100 ms)")